# Can a mathematical route loop back to an earlier date?

**The idea in everyday language:** Imagine two special regions of space-time. In a published mathematical example, a traveler passes through both and can return to an earlier event while their own clock keeps ticking forward. The equations describe a possible shape of space-time; they do not show how to build those regions or a ship that can travel through them.

This calculator checks when a mathematical loop can close and estimates where an ancient Earth event lies in the traveler’s starting frame. The first scale plot is a deliberately simple one-axis illustration; the later encounter calculation uses JPL Earth–Sun vectors and a stated local Galactic-motion model. Neither is a route to a historical witness.

**What the numbers mean:** they follow from the chosen equations and inputs. They do not prove that A.R.G.O. can make the trip. The Star story also leaves the exact date and viewing direction unknown.

## The calculation in detail

This chapter reproduces the symmetric closure benchmark discussed by Shoshany and Snodgrass and evaluates a separate flat 1+1 endpoint relation. The endpoint equation is a kinematic extension, not the paper's compact-support metric retargeted to Bethlehem.

The endpoint frame $S$ is the departure-comoving inertial frame used by the published construction: the traveler starts at rest there. A consistent target transform must include Earth's orbital velocity in A.R.G.O.'s initial state. In a one-axis illustration, assume the Sun's Galactic velocity and Earth's orbital velocity are collinear, transform the historical Earth event from a Galactic frame into $S$, and carry projected bounds for orbital phase, ground-site rotation, and Galactic curvature. This is a scenario calculation, not an ephemeris reconstruction.

For the paper's illustrative drive parameters, the transformed ancient Earth event is about $0.202$ light-years away in the chosen projection. The flat endpoint algebra still asks for a first leg of about $16{,}558$ years and $38{,}963$ light-years. That galactic-scale result diagnoses the endpoint extension as unsuitable for navigation; it is not a Bethlehem route or a solution of the paper's full metric.

The Star narrative supplies no exact epoch, range, or observer worldline. Treat 6 BCE only as a scenario label and explore a family of endpoint states.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display

# Parameters of the paper's explicit smooth-profile example.
U0, alpha0, omega0 = 0.75, 10.0, 0.95
lookback_years = 2031.0  # Illustrative scale only; the source narrative gives no exact year.
solar_galactic_speed_km_s = 230.0
earth_orbital_speed_km_s = 29.78
earth_rotation_speed_bound_km_s = 0.465  # Equatorial upper bound for a surface site.
galactocentric_radius_kpc = 8.2
earth_orbit_offset_bound_au = 2.0
seconds_per_year = 365.25 * 24 * 3600
meters_per_lightyear = 9.4607304725808e15
meters_per_au = 149_597_870_700.0
meters_per_kpc = 3.085677581491367e19
c_km_s = 299_792.458

# One-axis Galactic-frame to departure-rest-frame transform.
beta_sun = solar_galactic_speed_km_s / c_km_s
beta_earth_orbit = earth_orbital_speed_km_s / c_km_s
beta_departure = (beta_sun + beta_earth_orbit) / (1 + beta_sun * beta_earth_orbit)
gamma_departure = 1 / np.sqrt(1 - beta_departure**2)

def endpoint_transform(lookback):
    dt_galactic = -lookback
    dz_galactic = -beta_sun * lookback  # Shared linear Galactic drift; same solar-orbit phase assumed centrally.
    dt_departure = gamma_departure * (dt_galactic - beta_departure * dz_galactic)
    dz_departure = gamma_departure * (dz_galactic - beta_departure * dt_galactic)
    return dt_departure, dz_departure

def endpoint_coefficients(U, V):
    return (2 * V - U * (1 + V**2)) / (V - U), (V * U - 1) / (V - U)

V0 = U0 * (1 + omega0 * (alpha0 - 1) / 4)
A0, B0 = endpoint_coefficients(U0, V0)
t_target, z_target_ly = endpoint_transform(lookback_years)
galactic_acceleration = (solar_galactic_speed_km_s * 1000)**2 / (galactocentric_radius_kpc * meters_per_kpc)
galactic_curvature_bound_ly = 0.5 * galactic_acceleration * (lookback_years * seconds_per_year)**2 / meters_per_lightyear
earth_orbit_phase_bound_ly = earth_orbit_offset_bound_au * meters_per_au / meters_per_lightyear
earth_rotation_drift_bound_ly = earth_rotation_speed_bound_km_s / c_km_s * lookback_years
target_offset_bound_ly = galactic_curvature_bound_ly + earth_orbit_phase_bound_ly + earth_rotation_drift_bound_ly
z_target_min_ly, z_target_max_ly = z_target_ly - target_offset_bound_ly, z_target_ly + target_offset_bound_ly
T_first = (t_target - B0 * z_target_ly) / A0
D_first = V0 * T_first
T_orbit_bounds = sorted([(t_target - B0 * z_target_min_ly) / A0,
                         (t_target - B0 * z_target_max_ly) / A0])
print(f"V = {V0:.6f} c")
print(f"Ucrit = {2*V0/(1+V0**2):.8f}; chosen U = {U0:.3f}")
print(f"Tfinish/T2 = {A0:.8f}")
print(f"Departure speed in Galactic frame (collinear assumption) = {beta_departure*c_km_s:.3f} km/s")
print(f"Transformed target event: t = {t_target:,.6f} y, z = {z_target_ly:.6f} ly")
print(f"Projected target-position sensitivity = ±{target_offset_bound_ly:.6f} ly")
print(f"Flat endpoint estimate: T2 = {T_first:,.2f} y, first-leg separation = {D_first:,.2f} ly")
print(f"Symmetric-family lower bound for this target endpoint: D > {abs(t_target)-abs(z_target_ly):,.2f} ly")
print(f"First-leg time range under projected offset bounds: {T_orbit_bounds[0]:,.3f}–{T_orbit_bounds[1]:,.3f} y")
closure_speed_fraction_c = z_target_ly / t_target
closure_proper_time_years = np.sqrt(t_target**2 - z_target_ly**2)
print(f"Straight target-to-departure segment: timelike={abs(z_target_ly) < abs(t_target)}, velocity={closure_speed_fraction_c*c_km_s:.2f} km/s, proper time={closure_proper_time_years:,.4f} y")
print("Endpoint check only: tangent matching to the return-bubble exit is not solved.")

## Closed-loop threshold versus average bubble speed

In [ ]:
V_grid = np.linspace(1.001, 8.0, 500)
Ucrit_grid = 2 * V_grid / (1 + V_grid**2)
fig = go.Figure()
fig.add_trace(go.Scatter(x=V_grid, y=Ucrit_grid, name="minimum frame speed", mode="lines"))
fig.add_hline(y=U0, line_dash="dash", annotation_text=f"example U={U0}")
fig.add_vline(x=V0, line_dash="dot", annotation_text=f"example V={V0:.3f}")
fig.update_layout(title="Symmetric CTC threshold", xaxis_title="Average bubble separation speed V / c", yaxis_title="Required relative frame speed Ucrit / c", yaxis_range=[0,1], template="plotly_white")
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

## Flat endpoint extension: a scale diagnostic, not a mission trajectory

The plotted target event is transformed from a one-axis Galactic frame to the departure-comoving frame. The central scenario assumes collinear solar Galactic and Earth-orbital velocities and equal heliocentric orbital phase at the two epochs. The shaded interval adds projected Earth orbital-phase, surface-rotation, and Galactic-curvature scales; it is not a statistical error bar or complete uncertainty budget. This calculation does not reconstruct Earth's state or locate a Bethlehem observer. It does not integrate a Galactic potential, retarget the published finite-support metric, or solve the full geodesic. The thousands-of-light-years output shows that this endpoint extension is outside a credible local navigation regime.

**Try it:** Move the slider to choose a lookback time. The marker follows the model's central distance estimate. The shaded band is the same projected sensitivity scale described above. The year is a scenario input, not a date established by the Star narrative.

For the symmetric two-leg algebra, a target endpoint timelike-separated from departure has the necessary bound $D>|t_C|-|z_C|$. The dashed lower-bound curve applies to the central one-axis target transform; it is not a bound on every point of the published closed geodesic. The 100 AU line is a solar-system-scale reference. The actual finite-support metric and tangent-matched route are not solved here.

In [ ]:
lookbacks = np.linspace(1, 3000, 480)
t_targets, z_targets = endpoint_transform(lookbacks)
galactic_curvature_bounds = 0.5 * galactic_acceleration * (lookbacks * seconds_per_year)**2 / meters_per_lightyear
earth_orbit_bounds = np.full_like(lookbacks, earth_orbit_phase_bound_ly)
earth_rotation_bounds = earth_rotation_speed_bound_km_s / c_km_s * lookbacks
target_bounds = galactic_curvature_bounds + earth_orbit_bounds + earth_rotation_bounds
T_center = (t_targets - B0 * z_targets) / A0
D_center = V0 * T_center
D_bound_a = V0 * ((t_targets - B0 * (z_targets - target_bounds)) / A0)
D_bound_b = V0 * ((t_targets - B0 * (z_targets + target_bounds)) / A0)
D_low = np.minimum(D_bound_a, D_bound_b)
D_high = np.maximum(D_bound_a, D_bound_b)
fig = go.Figure()
fig.add_trace(go.Scatter(x=lookbacks, y=D_low, mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=lookbacks, y=D_high, mode="lines", line=dict(width=0), fill="tonexty", fillcolor="rgba(31,119,180,0.18)", name="projected target-offset sensitivity"))
fig.add_trace(go.Scatter(x=lookbacks, y=D_center, mode="lines", name="departure-frame central estimate"))
D_floor = np.abs(t_targets) - np.abs(z_targets)
fig.add_trace(go.Scatter(x=lookbacks, y=D_floor, mode="lines", line=dict(dash="dash"), name="symmetric-family lower bound"))
slider_years = np.unique(np.r_[np.arange(1, 2501, 50), lookback_years]).astype(int)
slider_distances = np.interp(slider_years, lookbacks, D_center)
fig.add_trace(go.Scatter(x=[lookback_years], y=[D_first], mode="markers", marker=dict(size=12, color="#c23b22"), name="selected lookback", hovertemplate="Lookback: %{x:,} years<br>Central estimate: %{y:,.0f} ly<extra></extra>"))
steps = [dict(method="update", label=f"{year:,}", args=[{"x": [[year]], "y": [[distance]]}, {"title": f"Flat-model scale at {year:,} years lookback"}, [4]]) for year, distance in zip(slider_years, slider_distances)]
fig.update_layout(title=f"Flat-model scale at {lookback_years:,} years lookback", xaxis_title="Lookback in Galactic frame (years; scenario input)", yaxis_title="First-leg separation (light-years)", yaxis_type="log", template="plotly_white", shapes=[dict(type="line", xref="paper", x0=0, x1=1, y0=100*meters_per_au/meters_per_lightyear, y1=100*meters_per_au/meters_per_lightyear, line=dict(color="gray", dash="dot"))], annotations=[dict(x=1, xref="paper", y=100*meters_per_au/meters_per_lightyear, text="100 AU reference", showarrow=False, yshift=10)], sliders=[dict(active=int(np.argmin(abs(slider_years-lookback_years))), currentvalue=dict(prefix="Target lookback: "), pad=dict(t=35), steps=steps)])
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

## Retargeting gap

The one-axis transform repairs the most direct frame inconsistency but does not produce a navigable target state. Galactic and heliocentric velocities are not actually collinear; orbital phases, observer location, Earth's rotation, non-circular Galactic motion, and gravitational fields need vector ephemerides in declared frames. A precision route also needs the retargeted metric and a full closed timelike geodesic intersecting both encounter and departure events. Changing this flat endpoint does not move the finite-support profiles, recompute the lapse, or solve the geodesic. See the [candidate review](../temporal-drive-candidate.md).

## A continuous loop can reach the date—and still collide

The previous scale plot used a collinear one-axis transform. Here we first transform the 6 BCE Earth-center event as a vector using JPL Earth–Sun states and a stated local Galactic-motion approximation, then rotate the loop axis to point along that displacement. The explicit smooth two-bubble centerline is scaled so an interior point reaches that event. The traveler remains on one continuous, future-directed timelike curve; it does not jump between dates.

The calculation uses JPL Horizons Earth–Sun geometric vectors at 6 BCE October 17 (Julian calendar) and 2026 October 17 (TDB), then adds a local Galactic-motion approximation: 230 km/s circular speed, 8.2 kpc radius, and solar peculiar velocity from [Schönrich et al. (2010)](https://doi.org/10.1111/j.1365-2966.2010.16253.x). The vector states use the [Horizons API](https://ssd-api.jpl.nasa.gov/doc/horizons.html); the approximation is recorded in [`estimate_earth_target_state_3d.py`](../../scripts/estimate_earth_target_state_3d.py). The vector estimate puts the Earth center about 0.2024 light-years from the departure event in the departure rest frame. The vector lies almost exactly along the instantaneous heliocentric Earth-orbit velocity (about 0.00645° away), while it is 102.57° from the total Earth velocity including solar Galactic motion. This is the expected common-motion cancellation in the departure-rest transform. The plot shows both directions; the chosen loop axis is aligned to the target vector. The exact-profile centerline then intersects Earth’s modeled center, a collision in this scenario. The result is still not a route to a witness: Galactic motion is approximated, ancient Earth orientation/site motion is omitted, and the calculation does not model Earth gravity, the finite ship and bubble wall, or photons traveling from A.R.G.O. to an observer. In the source profile, the return centerline sidesteps in x and has y=0. At this event its x offset has returned to 0 after the 0.8-year sidestep, matching the target vector after rotating the loop’s local axis to that vector. This three-coordinate centerline match uses the Earth-center scenario, not a site or witness position; ancient Earth rotation and observer location remain unknown. The paper uses the transverse sidestep to keep the compact field supports apart, but this notebook does not re-prove their global non-overlap after retargeting. A safe appearance still requires a full 3D world-tube clearance calculation and a solved light path.

**Try it:** Change the frame speed, profile parameters, or scenario event. The plot shows the entire loop and marks the target crossing. The notebook also plots the transformed target vector, prints its velocity-axis decomposition, and reports the loop target-position residual so the geometry and numerical fit are visible. The arXiv paper establishes the analytic curve as a geodesic of its prescribed metric; this calculator evaluates that centerline and does not establish a realizable drive. [Source paper](https://arxiv.org/abs/2309.10072).


In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "scripts").is_dir():
    repo_root = next(parent for parent in Path.cwd().parents if (parent / "scripts").is_dir())
sys.path.insert(0, str(repo_root / "scripts"))
from solve_two_bubble_encounter import calculate_encounter, sample_worldline
from estimate_earth_target_state_3d import estimate as estimate_earth_target

target_3d = estimate_earth_target()
t_target = float(target_3d["target_time_in_departure_frame_years"])
z_target_ly = float(target_3d["target_distance_lightyears"])  # Choose the loop axis along the target vector.
encounter = calculate_encounter(
    target_time_years=t_target, target_z_lightyears=z_target_ly,
    frame_speed=U0, alpha=alpha0, omega=omega0,
)
path = sample_worldline(encounter, count=1800)
target_xyz = np.array(target_3d["target_position_departure_frame_icrf_lightyears"], dtype=float)
departure_velocity = np.array(target_3d["departure_velocity_icrf_km_s"], dtype=float)
vhat = departure_velocity / np.linalg.norm(departure_velocity)
earth_orbit_velocity = np.array(target_3d["earth_sun_state"]["departure_velocity_km_s"], dtype=float)
earth_orbit_hat = earth_orbit_velocity / np.linalg.norm(earth_orbit_velocity)
parallel_xyz = np.dot(target_xyz, vhat) * vhat
orbit_axis_xyz = z_target_ly * earth_orbit_hat
print("Earth-center scenario transformed into the departure rest frame:")
print(f"  t = {t_target:.6f} y; position ICRF-oriented = {target_xyz} ly")
print(f"  distance = {z_target_ly:.9f} ly; cross-track to current Earth-velocity axis = {target_3d['target_cross_track_from_departure_velocity_axis_lightyears']:.6f} ly")
print(f"  angle from total Earth departure-velocity axis = {target_3d['target_angle_from_departure_velocity_axis_degrees']:.3f} deg")
print(f"  angle from heliocentric Earth-orbit velocity = {target_3d['target_angle_from_earth_orbital_velocity_axis_degrees']:.5f} deg")
print(f"  cross-track from orbital-velocity axis = {target_3d['target_cross_track_from_earth_orbital_velocity_axis_lightyears']:.3e} ly")

fig_vector = go.Figure()
fig_vector.add_trace(go.Scatter3d(x=[0,target_xyz[0]], y=[0,target_xyz[1]], z=[0,target_xyz[2]], mode="lines+markers", name="6 BCE Earth-center vector"))
fig_vector.add_trace(go.Scatter3d(x=[0,orbit_axis_xyz[0]], y=[0,orbit_axis_xyz[1]], z=[0,orbit_axis_xyz[2]], mode="lines+markers", name="Current heliocentric Earth-velocity direction", line=dict(dash="dot", color="#22c55e")))
fig_vector.add_trace(go.Scatter3d(x=[0,parallel_xyz[0]], y=[0,parallel_xyz[1]], z=[0,parallel_xyz[2]], mode="lines+markers", name="Projection along departure velocity", line=dict(dash="dash")))
fig_vector.add_trace(go.Scatter3d(x=[parallel_xyz[0],target_xyz[0]], y=[parallel_xyz[1],target_xyz[1]], z=[parallel_xyz[2],target_xyz[2]], mode="lines+markers", name="Transverse component", line=dict(width=7)))
fig_vector.update_layout(title="3D Earth-center displacement in the departure rest frame", scene=dict(xaxis_title="ICRF-oriented x (ly)", yaxis_title="ICRF-oriented y (ly)", zaxis_title="ICRF-oriented z (ly)", aspectmode="data"), height=560, legend=dict(orientation="h", y=-0.15))
fig_vector.show()

print(f"Loop target in its aligned axis: ({t_target:.6f} y, {z_target_ly:.9f} ly)")
print(f"Position residual: {encounter['target_coordinate_residual_lightyears']:.3e} ly")
print(f"Return-leg transition fraction: {encounter['return_transition_fraction']:.6f}")
print(f"Return centerline at target: x={encounter['return_centerline_x_at_target_lightyears']:.3e} ly, y={encounter['return_centerline_y_at_target_lightyears']:.3e} ly")
print(f"Return sidestep complete: {encounter['return_sidestep_complete_at_target']}")
print(f"First-leg separation: {encounter['first_leg_separation_lightyears']:,.1f} ly")
print(f"Traveler proper time to target: {encounter['traveler_proper_time_at_target_years']:,.1f} y")
print(f"Departure-frame coordinate derivative (not Earth-relative impact speed): {encounter['S_coordinate_velocity_z_fraction_c']*c_km_s:,.0f} km/s")
print("Safety result: Earth-center intersection — collision in this model.")

fig_loop = go.Figure()
fig_loop.add_trace(go.Scatter(x=path["outbound_t"], y=path["outbound_z"], mode="lines", name="First bubble leg"))
fig_loop.add_trace(go.Scatter(x=path["return_t"], y=path["return_z"], mode="lines", name="Second bubble leg"))
fig_loop.add_trace(go.Scatter(x=path["closure_t"], y=path["closure_z"], mode="lines", name="Continuous closing segment", line=dict(dash="dot")))
fig_loop.add_trace(go.Scatter(x=[t_target], y=[z_target_ly], mode="markers", name="6 BCE Earth-center event (collision)", marker=dict(size=12, color="#ef4444", symbol="x")))
fig_loop.update_layout(title="Explicit two-bubble loop through the modeled Earth-center event", xaxis_title="Departure-frame coordinate time (years)", yaxis_title="Displacement along modeled axis (light-years)", hovermode="closest", legend=dict(orientation="h", y=-0.25), height=560)
fig_loop.update_xaxes(showgrid=True, zeroline=True)
fig_loop.update_yaxes(showgrid=True, zeroline=True)
fig_loop.show()


## One journey, three views: 2D, 3D, and 4D

A journey through spacetime is a **worldline**: one continuous curve through space and time. We can draw the same curve in several ways. A 2D spacetime diagram uses coordinate time and distance along the route. A 3D space view shows where the ship is in space as the journey proceeds. A 4D spacetime event needs four numbers—time and three space coordinates—so a screen cannot show it all at once. We use a 3D projection and linked views instead.

In this example, the modeled route axis is called *z*. The small *x* movement is the sideways support sidestep from the paper; *y* is zero by construction. The illustrative 3D spacetime view (t, z, x) treats the plotted sideways offset as a visual extension; it is not derived by solving the full four-dimensional geodesic. The source construction guarantees the stated 1+1 centerline geodesic, but a valid 3+1 extension must solve the geodesic equations and verify position and tangent continuity through every transition. This plot does not represent the finite hull or bubble volume. A full 3D route would need all four coordinates, shown through linked projections such as (t, x), (t, y), (t, z), and (x, y), or through 3D space slices selected by τ.

Move the slider to advance the traveler’s proper time τ. The numerical clock is integrated segment by segment from the paper’s lapse profile, the Lorentz-factor coast, and the stationary closing segment; the slider value is now physical within this prescribed test-particle metric. The same traveler may see departure-frame coordinate time decrease while τ increases. This is still the paper’s idealized centerline, not an engineered ship or a complete Earth-centered mission model. The transverse sidestep is only about 1 km beside a route spanning thousands of light-years, so the 3D spacetime panel stretches that axis for visibility; its coordinates and aspect are not to scale. [Source equations, especially (3.14)–(3.20)](https://arxiv.org/abs/2309.10072).

In [ ]:
from plotly.subplots import make_subplots

# Concatenate in traveler order. τ is integrated from the prescribed metric profile.
route_t = np.array(path['outbound_t'] + path['return_t'] + path['closure_t'])
route_tau = np.array(path['outbound_tau'] + path['return_tau'] + path['closure_tau'])
route_x = np.array(path['outbound_x'] + path['return_x'] + path['closure_x'])
route_z = np.array(path['outbound_z'] + path['return_z'] + path['closure_z'])
route_y = np.zeros_like(route_x)  # This published centerline is planar by construction.
n = len(route_t)
frames_n = 64
tau_frames = np.linspace(route_tau[0], route_tau[-1], frames_n)
t_frame = np.interp(tau_frames, route_tau, route_t)
x_frame = np.interp(tau_frames, route_tau, route_x)
z_frame = np.interp(tau_frames, route_tau, route_z)
frame_names = [str(k) for k in range(frames_n)]

fig_path = make_subplots(rows=1, cols=2, specs=[[{'type':'xy'}, {'type':'scene'}]],
                        subplot_titles=('2D spacetime: (t, z)', '3D spacetime projection: (t, z, x)'))
fig_path.add_trace(go.Scatter(x=path['outbound_t'], y=path['outbound_z'], mode='lines', name='First leg'), row=1, col=1)
fig_path.add_trace(go.Scatter(x=path['return_t'], y=path['return_z'], mode='lines', name='Second leg'), row=1, col=1)
fig_path.add_trace(go.Scatter(x=path['closure_t'], y=path['closure_z'], mode='lines', name='Closing segment', line=dict(dash='dot')), row=1, col=1)
fig_path.add_trace(go.Scatter(x=[t_frame[0]], y=[z_frame[0]], mode='markers', name='Ship at τ', marker=dict(size=11, color='#f97316')), row=1, col=1)
fig_path.add_trace(go.Scatter3d(x=route_t, y=route_z, z=route_x, mode='lines', name='Same continuous worldline', line=dict(width=5, color='#2563eb')), row=1, col=2)
fig_path.add_trace(go.Scatter3d(x=[t_frame[0]], y=[z_frame[0]], z=[x_frame[0]], mode='markers', name='Ship at τ', marker=dict(size=6, color='#f97316')), row=1, col=2)
fig_path.frames = [go.Frame(name=frame_names[k], data=[
    go.Scatter(x=[t_frame[k]], y=[z_frame[k]]),
    go.Scatter3d(x=[t_frame[k]], y=[z_frame[k]], z=[x_frame[k]])
], traces=[3, 5]) for k in range(frames_n)]
slider_steps = [dict(method='animate', args=[[frame_names[k]], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))], label=f'{tau_frames[k]:,.0f}') for k in range(frames_n)]
fig_path.update_layout(title='Follow one continuous route as onboard proper time advances', height=650, template='plotly_white',
    sliders=[dict(active=0, currentvalue=dict(prefix='Traveler proper time τ (years): '), steps=slider_steps, pad=dict(t=45))],
    scene=dict(xaxis_title='Departure-frame coordinate time t (years)', yaxis_title='Route coordinate z (light-years)', zaxis_title='Sidestep x (light-years)', aspectmode='manual', aspectratio=dict(x=1.5,y=1,z=0.35)))
fig_path.update_xaxes(title_text='Departure-frame coordinate time t (years)', row=1, col=1)
fig_path.update_yaxes(title_text='Route coordinate z (light-years)', row=1, col=1)
fig_path.show()

# Spatial projection: remove t from the axes; use the same proper-time sequence.
fig_space = go.Figure()
fig_space.add_trace(go.Scatter3d(x=route_x, y=route_y, z=route_z, mode='lines', name='Spatial projection', line=dict(width=5, color='#2563eb')))
fig_space.add_trace(go.Scatter3d(x=[x_frame[0]], y=[0], z=[z_frame[0]], mode='markers', name='Ship at τ', marker=dict(size=7, color='#f97316')))
fig_space.frames = [go.Frame(name=frame_names[k], data=[go.Scatter3d(x=[x_frame[k]], y=[0], z=[z_frame[k]])], traces=[1]) for k in range(frames_n)]
space_steps = [dict(method='animate', args=[[frame_names[k]], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))], label=f'{tau_frames[k]:,.0f}') for k in range(frames_n)]
fig_space.update_layout(title='3D space view: the same traveler proper-time cursor', height=520, template='plotly_white',
    sliders=[dict(active=0, currentvalue=dict(prefix='Traveler proper time τ (years): '), steps=space_steps, pad=dict(t=45))],
    scene=dict(xaxis_title='x (light-years)', yaxis_title='y (light-years)', zaxis_title='z (light-years)', aspectmode='data'))
fig_space.show()

# Endpoint check: the modeled joins match in exterior coordinates and four-velocity.
gamma0 = 1.0 / np.sqrt(1.0-U0**2)
u0 = U0
out_event = np.array([path['outbound_t'][-1], path['outbound_x'][-1], 0.0, path['outbound_z'][-1]])
ret_start = np.array([path['return_t'][0], path['return_x'][0], 0.0, path['return_z'][0]])
ret_end = np.array([path['return_t'][-1], path['return_x'][-1], 0.0, path['return_z'][-1]])
close_start = np.array([path['closure_t'][0], path['closure_x'][0], 0.0, path['closure_z'][0]])
departure = np.zeros(4)
join_tangents = [
    ('outbound → return', np.array([gamma0,0,0,gamma0*u0]), np.array([gamma0,0,0,gamma0*u0])),
    ('return → closure', np.array([1,0,0,0]), np.array([1,0,0,0])),
    ('closure → departure', np.array([1,0,0,0]), np.array([1,0,0,0])),
]
join_events = [
    ('outbound → return', out_event, ret_start),
    ('return → closure', ret_end, close_start),
    ('closure → departure', np.array([path['closure_t'][-1],0.0,0.0,0.0]), departure),
]
print('Join check in exterior coordinates (t, x, y, z), c=1:')
for (name, left, right), (_, tangent_left, tangent_right) in zip(join_events, join_tangents):
    print(f'  {name}: max event-coordinate residual = {np.max(np.abs(left-right)):.3e}; four-velocity jump = {np.linalg.norm(tangent_left-tangent_right):.3e}')
print(f'Proper time increases across the sampled loop: {np.all(np.diff(route_tau) >= 0)}; total = {route_tau[-1]:,.1f} years.')


## A sighting needs a future light ray

The traveler reaching an ancient coordinate is not enough to make it visible. Light must leave the craft’s finite hull and reach a witness at a later event. In a flat local region, the exterior part of a radial ray satisfies

$$\Delta t = rac{L}{c}, \qquad \Delta t>0,$$

where *L* is the distance from the outer edge of the modeled field support to the observer. The graph below is a light-cone sanity check for that exterior segment only. Its slider changes the assumed range; it does not assign a distance to the Star of Bethlehem.

The model’s support radius is 1 km. This graph starts after a photon has crossed that boundary; it does not calculate propagation through the bubble interior or wall, the ship’s emission/reflection law, Earth’s curvature, or the witness’s historical motion. Also, the present trajectory targets Earth’s center, which the solver correctly flags as a collision. A sighting mission first needs a new, above-surface emission event and a specified observer worldline. The null-ray problem in a curved spacetime is $g_{\mu
u}k^\mu k^
u=0$ together with $k^
u
abla_
u k^\mu=0$, solved between those events in the same metric.

In [ ]:
# Exterior light-cone check in a local Minkowski patch (c = 1 ly/year here is not used;
# this small-scale plot uses km and seconds for readability).
c_km_s = 299_792.458
ranges_km = np.linspace(1.0, 1000.0, 32)
max_range_km = float(ranges_km[-1])
max_time_s = max_range_km / c_km_s * 1.05
time_s = np.linspace(0.0, max_time_s, 250)
cone_x_km = c_km_s * time_s
active_range_km = float(ranges_km[0])
arrival_s = active_range_km / c_km_s

fig_lightcone = go.Figure()
fig_lightcone.add_trace(go.Scatter(x=cone_x_km, y=time_s*1000.0, mode='lines', name='Outgoing light cone', line=dict(color='#f59e0b', width=3)))
fig_lightcone.add_trace(go.Scatter(x=-cone_x_km, y=time_s*1000.0, mode='lines', name='Opposite null direction', line=dict(color='#f59e0b', width=2, dash='dot')))
fig_lightcone.add_trace(go.Scatter(x=[active_range_km,active_range_km], y=[0,max_time_s*1000.0], mode='lines', name='Stationary observer worldline', line=dict(color='#2563eb', width=2)))
fig_lightcone.add_trace(go.Scatter(x=[0], y=[0], mode='markers', name='Photon crosses field boundary', marker=dict(color='#16a34a', size=10)))
fig_lightcone.add_trace(go.Scatter(x=[active_range_km], y=[arrival_s*1000.0], mode='markers', name='Null arrival event', marker=dict(color='#ef4444', size=11, symbol='x')))
fig_lightcone.frames = [go.Frame(name=str(i), data=[
    go.Scatter(x=[float(ranges_km[i]),float(ranges_km[i])], y=[0,max_time_s*1000.0]),
    go.Scatter(x=[float(ranges_km[i])], y=[float(ranges_km[i])/c_km_s*1000.0])
], traces=[2,4]) for i in range(len(ranges_km))]
lightcone_steps = [dict(method='animate', args=[[str(i)], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))], label=f'{ranges_km[i]:.0f}') for i in range(len(ranges_km))]
fig_lightcone.update_layout(title='The observer must lie on the emitter’s future light cone', xaxis_title='Distance from the field boundary (km)', yaxis_title='Local Minkowski time after boundary crossing (ms)', template='plotly_white', height=520,
    xaxis=dict(range=[-max_range_km*0.05,max_range_km*1.05]), yaxis=dict(range=[0,max_time_s*1000.0]),
    sliders=[dict(active=0,currentvalue=dict(prefix='Assumed exterior range L (km): '),steps=lightcone_steps,pad=dict(t=45))])
fig_lightcone.show()
print(f'Flat-patch exterior light time: 100 km → {100/c_km_s*1000:.4f} ms; 1,000 km → {1000/c_km_s*1000:.4f} ms.')
print('This is an exterior propagation baseline, not a predicted Bethlehem sighting.')

## Four events that a sighting mission must keep separate

A navigation plan needs four different places in its event ledger:

- $E_C$: Earth’s center at a chosen historical epoch.
- $E_W$: an event on a witness’s surface worldline.
- $E_{\mathrm{em}}$: an emission or reflection event on A.R.G.O.’s finite hull.
- $E_{\mathrm{rec}}$: the later event where the light reaches the witness.

The ship and Earth must not occupy the same event: their finite world-tubes must stay clear. The sighting condition is that a future-directed null ray starts at $E_{\mathrm{em}}$ and ends at $E_{\mathrm{rec}}$ on the witness worldline. A plotted ship marker at an ancient date does not satisfy this condition by itself.

The diagram is a separate, local scenario: a craft is placed directly above a surface observer at an assumed altitude *h*. The field support has the model’s 1 km radius. We show only the straight exterior segment from the near side of that support to the observer, assuming a flat local patch and vacuum. The slider does **not** retarget the closed loop or specify a Bethlehem location, orientation, emission mechanism, or date.

In [ ]:
# Radial local schematic: altitude is an explicit scenario input, not a solved route.
c_km_s = 299_792.458
field_radius_km = float(encounter.get('support_radius_km', 1.0))
altitude_km = np.linspace(2.0, 500.0, 36)
angles = np.linspace(0.0, 2*np.pi, 120)
selected_h = float(altitude_km[7])

fig_events = go.Figure()
fig_events.add_trace(go.Scatter(x=[-3,3], y=[0,0], mode='lines', name='Local ground / observer altitude', line=dict(color='#475569', width=4)))
fig_events.add_trace(go.Scatter(x=field_radius_km*np.cos(angles), y=selected_h+field_radius_km*np.sin(angles), mode='lines', name='1 km field-support boundary', line=dict(color='#8b5cf6', width=2, dash='dash'))) 
fig_events.add_trace(go.Scatter(x=[0], y=[selected_h], mode='markers', name='A.R.G.O. centerline event', marker=dict(color='#2563eb', size=12, symbol='diamond')))
fig_events.add_trace(go.Scatter(x=[0], y=[0], mode='markers', name='Surface observer event', marker=dict(color='#16a34a', size=12)))
fig_events.add_trace(go.Scatter(x=[0,0], y=[max(0,selected_h-field_radius_km),0], mode='lines+markers', name='Exterior null segment only', line=dict(color='#f59e0b', width=3), marker=dict(size=6)))
frames=[]
for i,h in enumerate(altitude_km):
    exterior=max(0.0,float(h)-field_radius_km)
    light_ms=exterior/c_km_s*1000.0
    frames.append(go.Frame(name=str(i), data=[
        go.Scatter(x=field_radius_km*np.cos(angles), y=float(h)+field_radius_km*np.sin(angles)),
        go.Scatter(x=[0], y=[float(h)]),
        go.Scatter(x=[0,0], y=[exterior,0])
    ], traces=[1,2,4], layout=go.Layout(title_text=f'Above-surface scenario: h = {h:.1f} km; exterior range = {exterior:.1f} km; flat-patch flight = {light_ms:.4f} ms')))
fig_events.frames=frames
event_steps=[dict(method='animate',args=[[str(i)],dict(mode='immediate',frame=dict(duration=0,redraw=True),transition=dict(duration=0))],label=f'{altitude_km[i]:.0f}') for i in range(len(altitude_km))]
fig_events.update_layout(title='Above-surface scenario',xaxis_title='Horizontal offset from observer (km; schematic)',yaxis_title='Height above local ground (km)',template='plotly_white',height=560,
    xaxis=dict(range=[-3,3]),yaxis=dict(range=[-2,505]),
    sliders=[dict(active=7,currentvalue=dict(prefix='Assumed A.R.G.O. center altitude h (km): '),steps=event_steps,pad=dict(t=45))])
fig_events.show()
h_example=100.0
print(f'At assumed h={h_example:.0f} km, exterior segment is {h_example-field_radius_km:.0f} km and flat-patch light time is {(h_example-field_radius_km)/c_km_s*1000:.4f} ms.')
print('Hull emission, propagation through the field wall, Earth orientation, and the actual witness worldline are not modeled here.')

## Retarget the loop away from Earth’s center (one explicit scenario)

The center-hit calculation is a collision benchmark, not the sighting route. Here is a first retargeting step: choose a hypothetical point 100 km above a spherical Earth, on an assumed radial direction parallel to the Earth-center displacement vector, and transform that event offset into the departure frame. The 6,471 km center-to-point vector is defined as simultaneous in the source ICRF/Galactic inertial frame. Its departure-frame time component is not generally zero; this does not reconstruct an Earth-fixed site. The code keeps that time shift when it fits the same closed-loop family to the new event.

This is a coordinate-consistent *scenario*, not a Bethlehem site reconstruction. The radial direction is deliberately chosen for illustration; ancient Earth orientation, observer location, site velocity, Earth’s full gravity, and the route’s clearance from Earth over the rest of the loop are still missing. Matching one above-surface event does not certify a safe trajectory.

In [ ]:
from estimate_earth_target_state_3d import transform_simultaneous_spatial_offset_to_departure

# Scenario inputs: no ancient site orientation is claimed.
earth_radius_km = 6371.0
assumed_altitude_km = 100.0
radial_distance_km = earth_radius_km + assumed_altitude_km
center_event_galactic = np.array(target_3d['target_position_galactic_icrf_km'], dtype=float)
radial_hat_galactic = center_event_galactic / np.linalg.norm(center_event_galactic)
offset_galactic_km = radial_distance_km * radial_hat_galactic
delta_t_surface_s, delta_r_surface_km = transform_simultaneous_spatial_offset_to_departure(
    tuple(offset_galactic_km), tuple(departure_velocity))
offset_time_years = delta_t_surface_s / seconds_per_year
offset_position_ly = np.array(delta_r_surface_km) / (meters_per_lightyear/1000.0)
surface_event_t = t_target + offset_time_years
surface_event_xyz = target_xyz + offset_position_ly
surface_event_range_ly = float(np.linalg.norm(surface_event_xyz))

surface_encounter = calculate_encounter(
    target_time_years=surface_event_t,
    target_z_lightyears=surface_event_range_ly,
    frame_speed=U0, alpha=alpha0, omega=omega0,
)
print('Above-surface event retarget (scenario only):')
print(f'  Assumed source-frame radial offset = {radial_distance_km:,.1f} km')
print(f'  Lorentz-transformed event offset: Δt_S = {delta_t_surface_s*1e6:.3f} μs, |Δr_S| = {np.linalg.norm(delta_r_surface_km):,.3f} km')
print(f'  target event: t = {surface_event_t:.9f} y; distance from departure = {surface_event_range_ly:.12f} ly')
print(f'  loop fit residuals: Δt = {surface_encounter["target_coordinate_residual_years"]:.3e} y, Δz = {surface_encounter["target_coordinate_residual_lightyears"]:.3e} ly')
print(f'  first leg = {surface_encounter["first_leg_separation_lightyears"]:,.1f} ly / {surface_encounter["T2_first_leg_years"]:,.2f} y')
print(f'  closure event time = {surface_encounter["loop_finish_time_S_years"]:,.2f} y; complete-loop construction retained.')
print('Earth orientation and whole-route collision clearance are not solved by this endpoint retarget.')

## 3D kinematic joins and the missing surface-site tangent

The paper prescribes a 3+1 metric and a closed timelike geodesic, but this calculator evaluates only its 1+1 time/route-axis projection. The plotted transverse offset is a separate smooth interpolation, so this particular 3D curve still needs a geodesic derivation. The cell below estimates one-sided tangents directly from finely sampled positions at each segment join; it no longer assumes the tangent values it is meant to check. Small residuals establish only that this particular plotted curve joins smoothly at the chosen sampling resolution. They do not establish the geodesic equation, finite-body clearance, or source-field consistency in 3+1 dimensions.

At departure, the model gives the traveler zero spatial velocity in the Earth-center departure-comoving frame. A surface site generally rotates relative to Earth’s center. Its speed is $v_{\rm rot}=0.465\cos\phi$ km/s for latitude $\phi$ in the spherical-Earth approximation, while its direction depends on longitude and epoch. A real launch therefore needs a site event and a matching four-velocity, or an explicit launch segment. Adding a position offset alone cannot supply that match.

In [ ]:
# Estimate position and one-sided tangent continuity from the sampled curve itself.
audit_path = sample_worldline(surface_encounter, count=50001)
segments = {
    'outbound': ('outbound_t','outbound_x','outbound_z'),
    'return': ('return_t','return_x','return_z'),
    'closure': ('closure_t','closure_x','closure_z'),
}
def endpoint_event(data, keys, side):
    i = 0 if side == 'start' else -1
    return np.array([data[keys[0]][i], data[keys[1]][i], 0.0, data[keys[2]][i]])
def endpoint_tangent(data, keys, side):
    i,j = (0,1) if side == 'start' else (-2,-1)
    dt = data[keys[0]][j] - data[keys[0]][i]
    return np.array([(data[keys[1]][j]-data[keys[1]][i])/dt, 0.0, (data[keys[2]][j]-data[keys[2]][i])/dt])
for left_name,left_side,right_name,right_side in [('outbound','end','return','start'),('return','end','closure','start'),('closure','end','outbound','start')]:
    lk,rk = segments[left_name],segments[right_name]
    event_gap = np.max(np.abs(endpoint_event(audit_path,lk,left_side)-endpoint_event(audit_path,rk,right_side)))
    tangent_gap = np.linalg.norm(endpoint_tangent(audit_path,lk,left_side)-endpoint_tangent(audit_path,rk,right_side))
    print(f'{left_name} → {right_name}: max event-coordinate gap = {event_gap:.3e} (y or ly); one-sided coordinate-tangent mismatch = {tangent_gap:.3e}')
print('These are finite-difference checks of the plotted curve only; they do not test ∇_u u = 0 in a specified 3+1 metric.')

# Surface-site rotational speed in the Earth-center instantaneous rest frame.
site_latitude_deg = np.linspace(-90.0,90.0,361)
site_rotation_km_s = earth_rotation_speed_bound_km_s*np.abs(np.cos(np.deg2rad(site_latitude_deg)))
fig_site = go.Figure(go.Scatter(x=site_latitude_deg,y=site_rotation_km_s,mode='lines',name='Surface-site speed'))
fig_site.add_hline(y=0,line_color='gray',line_dash='dot')
fig_site.update_layout(title='A grounded ship must match the site’s departure-frame velocity',xaxis_title='Assumed site latitude (degrees)',yaxis_title='Rotational speed relative to Earth center (km/s)',template='plotly_white',height=420)
fig_site.show()
for lat in [0,30,45,60,90]: print(f'latitude {lat:>2}°: site speed = {earth_rotation_speed_bound_km_s*np.cos(np.deg2rad(lat)):.3f} km/s ({earth_rotation_speed_bound_km_s*np.cos(np.deg2rad(lat))/c_km_s:.3e} c)')
print('Longitude, local orientation, site elevation, and the actual Astrarium location are unspecified.')

In [ ]:
# Certain endpoint collision in the current center-origin model.
surface_path = sample_worldline(surface_encounter, count=1001)
start_xyz_km = np.array([surface_path['outbound_x'][0], 0.0, surface_path['outbound_z'][0]]) * (meters_per_lightyear/1000.0)
return_xyz_km = np.array([surface_path['closure_x'][-1], 0.0, surface_path['closure_z'][-1]]) * (meters_per_lightyear/1000.0)
earth_radius_km = 6371.0
print('Earth-center clearance at modeled departure:', f'{np.linalg.norm(start_xyz_km)-earth_radius_km:,.1f} km')
print('Earth-center clearance at loop return:     ', f'{np.linalg.norm(return_xyz_km)-earth_radius_km:,.1f} km')
print('The off-center ancient target does not move the CTC origin; this loop therefore fails a whole-route Earth-clearance screen.')

# First-order departure-frame sensitivity from an omitted site rotational velocity.
lat_deg = np.linspace(0.0,90.0,181)
site_rotation_km_s = earth_rotation_speed_bound_km_s*np.cos(np.deg2rad(lat_deg))
lookback_for_spin_y = float(target_3d['lookback_years_galactic'])
shift_bound_ly = site_rotation_km_s/c_km_s*lookback_for_spin_y
fig_spin = go.Figure()
fig_spin.add_trace(go.Scatter(x=lat_deg,y=shift_bound_ly,mode='lines',name='First-order frame sensitivity'))
fig_spin.update_layout(title='Departure-site rotation changes the historical target frame',xaxis_title='Assumed departure latitude (degrees)',yaxis_title='Approximate position scale v_rotation × lookback (light-years)',template='plotly_white',height=420)
fig_spin.show()
print(f'Equatorial upper-bound scale: {shift_bound_ly[0]:.6f} ly = {shift_bound_ly[0]*63241.077:.1f} AU over {lookback_for_spin_y:.2f} y.')
print('Direction and exact transformed target require departure-site coordinates, Earth orientation, and site velocity.')